# 基礎編11
このノートブックでは、ユーザーの情報取得等に関するAPIを使用したスマートコントラクトの例を示します。

### 使用するスマートコントラクトAPI
queryUserIdByWallet，queryUser，isValidWalletAddressFormat, getIdByAliasName

## スマートコントラクトの処理の概要

- 引数としてウォレットアドレスとユーザ名が与えられます。
- ウォレットアドレスの形式が不正な場合はエラー終了します。
- すでにウォレットアドレスが登録済みの場合にはエラー終了します。
- 与えられたユーザ名のユーザが存在しない場合には、新たに作成します。
- ウォレットアドレスをユーザに登録します。
- ユーザの詳細情報を取得し、返り値として返します。 

## 準備
実行の準備を行います。

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

## スマートコントラクトのデプロイ

スマートコントラクトを便宜的にfunctionとして記述します。

In [2]:
function basic11(walletAddress, username){

    // ウォレットアドレスの形式が不正な場合はエラー終了します。
    if (!isValidWalletAddressFormat(walletAddress)) {
        throw "Invalid wallet address";
    }

    //  すでにウォレットアドレスが登録済みの場合にはエラー終了します。
    var uid = queryUserIdByWallet(walletAddress);
    if(uid){
        throw "wallet address is registered with " + uid;
    }

    // 与えられたユーザ名のユーザが存在しない場合には、新たに作成します。
    var uid = getIdByAliasName(username);
    if(!uid){
        uid = openContract("c1create").call({
            type: "user",
            name: username,
            domain: getContractDomainId()
        });
    }

    // ウォレットアドレスをユーザに登録します
    openContract("c1update").call({ id: uid, prop: "add wallets", value: [walletAddress] });

    // ユーザの詳細情報を取得し、返り値として返します。 
    return queryUser(uid);
}

上記のスマートコントラクトをデプロイします。  
ここではユーティリティ関数を使い詳細な手順は省略します。（具体的手順は基礎編3を参照してください）

In [3]:
var cid = await deploySmartContract({ walletAddress: 'string', username: 'string'}, basic11);
console.log(cid);

c042140317


デプロイしたスマートコントラクトにドメインの管理権限を与えます。  
（c1createおよびc1updateが実行できるように）

In [4]:
await rpc.call(adminWallet, 'c1update', { id: '@'+domain, prop: 'add managed_by', value: [cid] });

{
  txno: 701793,
  txid: 'xusPw6RwMpQaR85sZ3ajYPkJe8pUzB5PyQJayLznrdobu',
  status: 'ok',
  value: null
}

## スマートコントラクトの実行

テスト用にランダムなユーザ名とウォレットを生成します。

In [5]:
var newUserName = "tst11u" + Math.floor(Math.random()*100000000);
var newWallet = await api.importSigningWallet('es', await api.generateWalletKey('es'));
console.log(newWallet.address, newUserName);

eB29AmdfcUjfGixyJhvpC54aXDQA4e tst11u68863570


スマートコントラクトを実行します。  
指定したウォレットアドレスに紐づけられたユーザが、指定したユーザ名で作成されます。

In [6]:
var resp = await rpc.call(adminWallet, cid, {
    walletAddress: newWallet.address,
    username: newUserName
});
console.log(resp);

{
  txno: 701794,
  txid: 'xX33Xf9NxuYb8CGDxQXGvvRT2PxcSqMTaYnepuokBVwN8',
  status: 'ok',
  value: {
    id: 'u23518703',
    name: 'tst11u68863570',
    domain: 'd93319890',
    description: '',
    multisig: 1,
    wallets: [ 'eB29AmdfcUjfGixyJhvpC54aXDQA4e' ],
    callable_to: [ 'all' ],
    disclosed_to: [],
    groups: []
  }
}


ウォレットアドレスの形式が不正な場合は、スマートコントラクトの実行はエラーになります。

In [7]:
var resp = await rpc.call(adminWallet, cid, {
    walletAddress: 'invalid+address',
    username: newUserName
});
console.log(resp);

{
  txno: 701797,
  txid: 'xPKsnuPxjrRLbDea4KmipneE7LLu8CmAiCtzXtC7hrRgAB',
  status: 'thrown',
  value: 'Invalid wallet address'
}


すでにウォレットアドレスが登録済みの場合にはエラー終了します。

In [8]:
var resp = await rpc.call(adminWallet, cid, {
    walletAddress: newWallet.address,
    username: 'another_user'
});
console.log(resp);

{
  txno: 701798,
  txid: 'xjZjHwsYSBgheQYA8vREFeprYkkFr3v8SqjSei9eJCBCp',
  status: 'thrown',
  value: 'wallet address is registered with u23518703'
}


別のウォレットを追加します。  
resp.value.walletsで、ウォレットアドレスが追加されていることが分かります。

In [9]:
var newWallet2 = await api.importSigningWallet('es', await api.generateWalletKey('es'));
console.log(newWallet2.address);
var resp = await rpc.call(adminWallet, cid, {
    walletAddress: newWallet2.address,
    username: newUserName
});
console.log(resp);

ey9B7pcW39EP6LYTKqx45gpvNv8KFv
{
  txno: 701799,
  txid: 'xetHDnE4ZTc4YPnqZE7KwjNovXcCLMjhCuKRe84D2dDM3',
  status: 'ok',
  value: {
    id: 'u23518703',
    name: 'tst11u68863570',
    domain: 'd93319890',
    description: '',
    multisig: 1,
    wallets: [
      'eB29AmdfcUjfGixyJhvpC54aXDQA4e',
      'ey9B7pcW39EP6LYTKqx45gpvNv8KFv'
    ],
    callable_to: [ 'all' ],
    disclosed_to: [],
    groups: []
  }
}
